# DataVint Quick Start (5 minutes)

The fastest way to get started with DataVint data quality checks.

In [2]:
# Setup
import sys
sys.path.insert(0, '..')

import datavint as dv

## 1. Profile Your Dataset (< 1 second)

Get a quick overview before running quality checks.

In [3]:
dv.profile_dataset(
    "../playground/raw_data/movielens_train.csv",
    label_col="label"
)


═══════════════════════════════════════════════════════════════
📊 Dataset Profile
═══════════════════════════════════════════════════════════════
📁 Source: movielens_train.csv
📏 Shape: 80,668 rows × 8 columns
💾 Memory: 9.2 MB

📋 Column Types:
   • Numeric: 6 columns
     userId, movieId, rating, year, month, ... (+1 more)
   • Categorical: 1 columns
     genre

⚠️  Missing Values:
   • No missing values ✅

🎯 Label Distribution (label):
   • Positive (1): 48.7% (39,254 samples)
   • Negative (0): 51.3% (41,414 samples)
   • Balance: Excellent ✅ (perfectly balanced)

📝 Sample Preview (first 5 rows):
 userId  movieId  label  rating     genre  year  month  user_activity
    429      595      1     5.0 Animation  1996      3             58
    429      588      1     5.0 Adventure  1996      3             58
    429      590      1     5.0 Adventure  1996      3             58
    429      592      1     5.0    Action  1996      3             58
    429      432      0     3.0 Adventure  1

## 2. Compare Train vs Test (< 1 second)

Check for distribution shifts before training.

In [4]:
dv.compare_datasets(
    train_data="../playground/raw_data/movielens_train.csv",
    test_data="../playground/raw_data/movielens_test.csv",
    label_col="label"
)


═══════════════════════════════════════════════════════════════
📊 Dataset Comparison: Train vs Test
═══════════════════════════════════════════════════════════════

                               Train            Test          Δ
───────────────────────────────────────────────────────────────
Rows                          80,668          20,168     -75.0%
Columns                            8               8          0
Memory                          9.2 MB            2.3 MB     -75.0%
Missing %                       0.0%            0.0%      +0.0%
Duplicates %                    0.0%            0.0%      +0.0%

Label (label):
  0                            51.3%           53.8%      +2.4%
  1                            48.7%           46.2%      -2.4%

⚠️  Large difference in row count (>50%)

Next: Run dv.detect_issues() with serving_statistics for detailed skew analysis


## 3. Generate Statistics (2-5 seconds)

Compute detailed per-feature statistics.

In [5]:
train_stats = dv.generate_statistics(
    "../playground/raw_data/movielens_train.csv",
    label_col="label"
)

test_stats = dv.generate_statistics(
    "../playground/raw_data/movielens_test.csv",
    label_col="label"
)

print(f"✅ Train: {train_stats.n_rows:,} rows")
print(f"✅ Test:  {test_stats.n_rows:,} rows")

[datavint] Loading data from ../playground/raw_data/movielens_train.csv
[datavint] Computing statistics for 80,668 rows, 8 columns
[datavint] Generated statistics for 7 features
[datavint] Loading data from ../playground/raw_data/movielens_test.csv
[datavint] Computing statistics for 20,168 rows, 8 columns
[datavint] Generated statistics for 7 features


✅ Train: 80,668 rows
✅ Test:  20,168 rows


## 4. Detect Quality Issues (< 1 second)

Run all 6 detectors automatically.

In [ ]:
issues = dv.detect_issues(
    statistics=train_stats,
    serving_statistics=test_stats  # Optional: enables train-test skew detection
)

dv.display_issues(issues)

## 5. Generate Manifest (v0.2 - Data Quality Optimization)

Generate a manifest to fix the detected issues automatically.

In [ ]:
# Generate manifest from detected issues
manifest = dv.generate_manifest(
    statistics=train_stats,
    serving_statistics=test_stats
)

print(f"✅ Manifest generated")
print(f"   Rows to keep: {manifest.row_mask.sum():,}/{len(manifest.row_mask):,} ({manifest.row_mask.sum()/len(manifest.row_mask)*100:.1f}%)")
print(f"   Sample weights: min={manifest.sample_weights.min():.2f}, max={manifest.sample_weights.max():.2f}")
print(f"   Feature fixes: {len(manifest.feature_fixes)} feature(s)")

if manifest.feature_fixes:
    print("\n   Fixes applied:")
    for feat, fix in manifest.feature_fixes.items():
        print(f"     - {feat}: {fix['method']} imputation")

## Apply Manifest to Training Data

Use the manifest to correct your training data before model training.

In [ ]:
# Load training data and apply manifest
import pandas as pd

train_df = pd.read_csv("../playground/raw_data/movielens_train.csv")
print(f"Original training data: {train_df.shape}")

# Apply manifest (out-of-place - returns corrected copy)
corrected_df = manifest.apply(train_df, inplace=False)
print(f"Corrected training data: {corrected_df.shape}")

# Check improvements
if '__datavint_weight__' in corrected_df.columns:
    print(f"\n✅ Sample weights added: {corrected_df['__datavint_weight__'].describe()[['min', 'max', 'mean']]}")

# Re-validate corrected data
corrected_stats = dv.generate_statistics(corrected_df, label_col="label")
corrected_issues = dv.detect_issues(corrected_stats, serving_statistics=test_stats)

print(f"\n📊 Quality improvement:")
print(f"   Before: {len(issues)} issues detected")
print(f"   After: {len(corrected_issues)} issues detected")
print(f"   Improvement: {len(issues) - len(corrected_issues)} issues fixed ✅")

## 6. Now Try With Anomalous Data

See what DataVint detects in a dataset with injected quality issues.

In [7]:
# Profile first
dv.profile_dataset(
    "../playground/raw_data/movielens_anomalous.csv",
    label_col="label"
)


═══════════════════════════════════════════════════════════════
📊 Dataset Profile
═══════════════════════════════════════════════════════════════
📁 Source: movielens_anomalous.csv
📏 Shape: 81,474 rows × 8 columns
💾 Memory: 9.1 MB

📋 Column Types:
   • Numeric: 6 columns
     userId, movieId, rating, year, month, ... (+1 more)
   • Categorical: 1 columns
     genre

⚠️  Missing Values:
   • Total: 4,082 missing cells (0.6%)
     - genre: 4,082 (5.0%)

🎯 Label Distribution (label):
   • Positive (1): 48.7% (39,670 samples)
   • Negative (0): 51.3% (41,804 samples)
   • Balance: Excellent ✅ (perfectly balanced)

📝 Sample Preview (first 5 rows):
 userId  movieId  label  rating     genre  year  month  user_activity
    429      595      1     5.0 Animation  1996      3           1211
    429      588      1     5.0 Adventure  1996      3             58
    429      590      1     5.0 Adventure  1996      3             58
    429      592      1     5.0    Action  1996      3             58

In [8]:
# Detect issues
anomalous_stats = dv.generate_statistics(
    "../playground/raw_data/movielens_anomalous.csv",
    label_col="label"
)

anomalous_issues = dv.detect_issues(anomalous_stats)

dv.display_issues(anomalous_issues)

[datavint] Loading data from ../playground/raw_data/movielens_anomalous.csv
[datavint] Computing statistics for 81,474 rows, 8 columns
[datavint] Generated statistics for 7 features


✅ No issues detected!


## That's It!

You now know the complete DataVint v0.2 workflow:

1. **Profile** → Quick dataset overview
2. **Compare** → Check train/test similarity
3. **Statistics** → Detailed feature analysis
4. **Detect** → Find quality issues
5. **Generate & Apply Manifest** → Automatically fix data quality issues (v0.2!)

**What manifests do:**
- 🎯 Filter bad rows (row_mask)
- ⚖️ Reweight samples (sample_weights)
- 🔧 Impute missing values (feature_fixes)

**Next steps:**
- Try `data_profiling_demo.ipynb` for detailed examples
- Use DataVint on your own datasets
- Check out the validation script: `validate_datavint.py`
- Read API docs in `docs/` folder